# Exercise 4 — train ResNet on Kaggle GPU

**Before running:**
1. In Kaggle: **Settings → Accelerator → GPU** (T4 or P100), and **Settings → Internet → On** (needed for `git clone` + pip).
2. Make sure your repo is pushed **including `model.py`** and the edited `data.py` / `train.py` / `trainer.py` / `export_onnx.py`.

The `.onnx` you upload to the leaderboard will end up in `/kaggle/working/` (downloadable from the notebook output).

In [ ]:
# 0. Sanity: GPU + torch
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
!nvidia-smi -L

In [ ]:
# 1. Clone the repo fresh into the writable working dir
%cd /kaggle/working
!rm -rf deep-learning
!git clone --depth 1 https://github.com/mo-karbalaee/deep-learning.git
%cd /kaggle/working/deep-learning/ex4/src_to_implement

In [ ]:
# 2. Dependencies. Kaggle already has torch/torchvision/scikit-image/pandas/scikit-learn.
#    onnxruntime runs the test suite; onnxscript is needed by torch 2.x's ONNX exporter.
!pip -q install onnxruntime onnxscript
import skimage, pandas, sklearn, torchvision
print('deps OK')

In [ ]:
# 3. Extract the dataset (images.zip is committed in the repo)
import zipfile, os
if not os.path.isdir('images'):
    zipfile.ZipFile('images.zip').extractall('.')
print(len(os.listdir('images')), 'images extracted')

In [ ]:
# 4. (Optional) run the unit tests / bonus breakdown
!python PytorchChallengeTests.py Bonus

In [ ]:
# 5. Train on the GPU. train.py uses cuda automatically when available,
#    creates checkpoints/, early-stops, and saves losses.png.
#    Edit epochs/batch_size/lr inside train.py if you want to tune.
!python train.py

In [ ]:
# 6. Export the best checkpoint to ONNX and copy it to /kaggle/working for download.
#    train.py only saves a checkpoint when the validation loss improves,
#    so the highest-numbered checkpoint is the best one.
import glob, os, re, shutil
ckpts = sorted(glob.glob('checkpoints/checkpoint_*.ckp'))
assert ckpts, 'No checkpoints found — did training run?'
best = ckpts[-1]
epoch = int(re.findall(r'(\d+)', os.path.basename(best))[0])
print('best checkpoint:', best, '| epoch', epoch)
!python export_onnx.py {epoch}
onnx_name = 'checkpoint_{:03d}.onnx'.format(epoch)
shutil.copy(onnx_name, '/kaggle/working/' + onnx_name)
if os.path.exists('losses.png'):
    shutil.copy('losses.png', '/kaggle/working/losses.png')
print('Saved /kaggle/working/' + onnx_name + '  — download it and upload to the leaderboard.')